In [210]:
import pandas as pd
import numpy as np
import warnings
from math import sqrt
warnings.filterwarnings('ignore')
from azureml.core.run import Run
from azureml.core.experiment import Experiment
from azureml.core.workspace import Workspace
from azureml.core.model import Model
from azureml.core.authentication import ServicePrincipalAuthentication
from azureml.train.automl import AutoMLConfig
from azureml.core import Workspace, Dataset
from sklearn.ensemble import RandomForestClassifier
import pickle
from matplotlib import pyplot as plt
from matplotlib.pyplot import figure
import mlflow
from azureml.data import DataType

In [211]:


subscription_id = 'f2f70602-65d9-479b-8014-a1c92a06ef0e'
resource_group = 'Learn_MLOps'
workspace_name = 'MLOps'

workspace = Workspace(subscription_id, resource_group, workspace_name)

In [212]:
uri = workspace.get_mlflow_tracking_uri()
mlflow.set_tracking_uri(uri)

In [213]:
# Importing pre-processed dataset
dataset = Dataset.get_by_name(workspace, name='processed_weather_data_portofTurku', version='latest')
print(dataset.name, dataset.version)

processed_weather_data_portofTurku 2


In [214]:
# Azure ML misreads int labels as boolean: True→1, False→0, NaN=snow(2)
for col in ['Current_weather_condition', 'Future_weather_condition']:                                 
    df[col] = df[col].map({True: 1, False: 0}).fillna(2).astype(int) 

In [215]:
df = dataset.to_pandas_dataframe()
print("Dataset version:", dataset.version)
print("\nColumn dtypes:")                                                                             
print(df.dtypes)
print("\nNaN counts:")                                                                                
print(df.isna().sum())


{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}
Dataset version: 2

Column dtypes:
Timestamp                    datetime64[ns]
Location                             object
Temperature_C                       float64
Humidity                            float64
Wind_speed_kmph                     float64
Wind_bearing_degrees                  int64
Visibility_km                       float64
Pressure_millibars                  float64
Current_weather_condition             int64
Future_weather_condition              int64
dtype: object

NaN counts:
Timestamp                    0
Location                     0
Temperature_C                0
Humidity                     0
Wind_speed_kmph              0
Wind_bearing_degrees         0
Visibility_km                0
Pressure_millibars           0
Current_weather_condition    0
Future_weather_condition     0
dtype: int64


In [216]:
df.head()

,Timestamp,Location,Temperature_C,Humidity,Wind_speed_kmph,Wind_bearing_degrees,Visibility_km,Pressure_millibars,Current_weather_condition,Future_weather_condition
0,2006-04-01 02:00:00,"Port of Turku, Finland",8.755556,0.83,11.0446,259,15.8263,1016.51,1,1
1,2006-04-01 03:00:00,"Port of Turku, Finland",9.222222,0.85,13.9587,258,14.9569,1016.66,1,1
2,2006-04-01 04:00:00,"Port of Turku, Finland",7.733333,0.95,12.3648,259,9.9820,1016.72,1,1
3,2006-04-01 05:00:00,"Port of Turku, Finland",8.772222,0.89,14.1519,260,9.9820,1016.84,1,1
4,2006-04-01 06:00:00,"Port of Turku, Finland",10.822222,0.82,11.3183,259,9.9820,1017.37,1,1


# Spliting Pre-Processed data into Training and Validation datasets

In [217]:
# Validation set is used later to evaluate model performance post training. 

In [218]:
df_training = df.iloc[:77160]

In [219]:
df_training.shape

(77160, 10)

In [220]:
df_validation = df.drop(df_training.index)

In [221]:
df_validation.shape

(19289, 10)

# Registering Training and Validation data to the datastore on the workspace. 

In [222]:
!mkdir Data

mkdir: cannot create directory ‘Data’: File exists


In [223]:
df_training.to_csv('Data/training_data.csv',index=False)

In [224]:
df_validation.to_csv('Data/validation_data.csv',index=False)

In [225]:
datastore = workspace.get_default_datastore()

In [226]:
datastore.upload(src_dir='Data', target_path='data', overwrite=True)

Uploading an estimated of 2 files
Uploading Data/validation_data.csv
Uploaded Data/validation_data.csv, 1 files out of an estimated total of 2
Uploading Data/training_data.csv
Uploaded Data/training_data.csv, 2 files out of an estimated total of 2
Uploaded 2 files


$AZUREML_DATAREFERENCE_12dee189b36a47bebce846e0f9aebc02

In [227]:
df = dataset.to_pandas_dataframe()
print("Dataset version:", dataset.version)
print("\nColumn dtypes:")                                                                             
print(df.dtypes)
print("\nNaN counts:")                                                                                
print(df.isna().sum())

{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


Dataset version: 2

Column dtypes:
Timestamp                    datetime64[ns]
Location                             object
Temperature_C                       float64
Humidity                            float64
Wind_speed_kmph                     float64
Wind_bearing_degrees                  int64
Visibility_km                       float64
Pressure_millibars                  float64
Current_weather_condition             int64
Future_weather_condition              int64
dtype: object

NaN counts:
Timestamp                    0
Location                     0
Temperature_C                0
Humidity                     0
Wind_speed_kmph              0
Wind_bearing_degrees         0
Visibility_km                0
Pressure_millibars           0
Current_weather_condition    0
Future_weather_condition     0
dtype: int64


In [228]:
training_dataset = Dataset.Tabular.from_delimited_files(                                              
      path=datastore.path('data/training_data.csv'),      
      set_column_types={                                                                                
          'Current_weather_condition': DataType.to_long(),
          'Future_weather_condition': DataType.to_long()  
      }                                                 
  )

In [229]:
validation_dataset = Dataset.Tabular.from_delimited_files(                                            
      path=datastore.path('data/validation_data.csv'),      
      set_column_types={                              
          'Current_weather_condition': DataType.to_long(),
          'Future_weather_condition': DataType.to_long()  
      }                                                                                                 
  )

In [230]:
training_ds = training_dataset.register(workspace=workspace,
                                 name='training_dataset',
                                 description='Dataset to use for ML training',
                                 create_new_version=True  # add this 
)

In [231]:
validation_ds = validation_dataset.register(workspace=workspace,
                                 name='validation_dataset',
                                 description='Dataset for validation ML models',
                                 create_new_version=True  # add this
                                 )

# Data ingestion step - Training dataset

In [232]:
dataset = Dataset.get_by_name(workspace, name='training_dataset')
print(dataset.name, dataset.version)

training_dataset 6


In [233]:
df = dataset.to_pandas_dataframe()
print("Dataset version:", dataset.version)
print("\nColumn dtypes:")                                                                             
print(df.dtypes)
print("\nNaN counts:")                                                                                
print(df.isna().sum())

{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}
Dataset version: 6

Column dtypes:
Timestamp                    datetime64[ns]
Location                             object
Temperature_C                       float64
Humidity                            float64
Wind_speed_kmph                     float64
Wind_bearing_degrees                  int64
Visibility_km                       float64
Pressure_millibars                  float64
Current_weather_condition             int64
Future_weather_condition              int64
dtype: object

NaN counts:
Timestamp                    0
Location                     0
Temperature_C                0
Humidity                     0
Wind_speed_kmph              0
Wind_bearing_degrees         0
Visibility_km                0
Pressure_millibars           0
Current_weather_condition    0
Future_weather_condition     0
dtype: int64


In [234]:
df.head()

,Timestamp,Location,Temperature_C,Humidity,Wind_speed_kmph,Wind_bearing_degrees,Visibility_km,Pressure_millibars,Current_weather_condition,Future_weather_condition
0,2006-04-01 02:00:00,"Port of Turku, Finland",8.755556,0.83,11.0446,259,15.8263,1016.51,1,1
1,2006-04-01 03:00:00,"Port of Turku, Finland",9.222222,0.85,13.9587,258,14.9569,1016.66,1,1
2,2006-04-01 04:00:00,"Port of Turku, Finland",7.733333,0.95,12.3648,259,9.9820,1016.72,1,1
3,2006-04-01 05:00:00,"Port of Turku, Finland",8.772222,0.89,14.1519,260,9.9820,1016.84,1,1
4,2006-04-01 06:00:00,"Port of Turku, Finland",10.822222,0.82,11.3183,259,9.9820,1017.37,1,1


In [235]:
df.shape

(77160, 10)

#### Feature Selection and scaling

In [236]:
X = df[['Temperature_C', 'Humidity', 'Wind_speed_kmph', 'Wind_bearing_degrees', 'Visibility_km', 'Pressure_millibars', 'Current_weather_condition']].values
y = df['Future_weather_condition'].values
y

array([1, 1, 1, ..., 1, 1, 1])

In [237]:
# Splitting the Training dataset into Train and Test set for ML training
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [238]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()

In [239]:
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# Model training and Testing Step

## 1. Support Vector Machine

In [240]:
myexperiment = Experiment(workspace, "support-vector-machine")
mlflow.set_experiment("mlflow-support-vector-machine")

<Experiment: artifact_location='', creation_time=1778021785804, experiment_id='5e64a8df-a05c-4986-a21a-0d5f24b998ca', last_update_time=None, lifecycle_stage='active', name='mlflow-support-vector-machine', tags={}>

In [241]:
#from sklearn.svm import SVC
from sklearn import svm, datasets
from sklearn.model_selection import GridSearchCV

In [242]:
parameters = {'kernel':('linear', 'rbf'), 'C':[1, 10]}

In [243]:
svc = svm.SVC()

In [244]:
# initialize a run in Azureml and mlflow experiments
run = myexperiment.start_logging()
mlflow.start_run()
# mlflow.end_run()


run.log("dataset name", dataset.name)
run.log("dataset Version", dataset.version)

Exception: Run with UUID c447bc73-d74a-494e-93b2-925978098359 is already active. To start a new run, first end the current run with mlflow.end_run(). To start a nested run, call start_run with nested=True

In [245]:
svc_grid = GridSearchCV(svc, parameters)

In [246]:
%%time
svc_grid.fit(X_train, y_train)

CPU times: user 24min 38s, sys: 2.03 s, total: 24min 40s
Wall time: 24min 40s


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",SVC()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [1, 10], 'kernel': ('linear', ...)}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",None
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fold and candidate parameter ind

In [247]:
svc_grid.get_params(deep=True)

{'cv': None,
 'error_score': nan,
 'estimator__C': 1.0,
 'estimator__break_ties': False,
 'estimator__cache_size': 200,
 'estimator__class_weight': None,
 'estimator__coef0': 0.0,
 'estimator__decision_function_shape': 'ovr',
 'estimator__degree': 3,
 'estimator__gamma': 'scale',
 'estimator__kernel': 'rbf',
 'estimator__max_iter': -1,
 'estimator__probability': False,
 'estimator__random_state': None,
 'estimator__shrinking': True,
 'estimator__tol': 0.001,
 'estimator__verbose': False,
 'estimator': SVC(),
 'n_jobs': None,
 'param_grid': {'kernel': ('linear', 'rbf'), 'C': [1, 10]},
 'pre_dispatch': '2*n_jobs',
 'refit': True,
 'return_train_score': False,
 'scoring': None,
 'verbose': 0}

In [248]:
from sklearn.svm import SVC

In [249]:
svc = SVC(C=svc_grid.get_params(deep=True)['estimator__C'], kernel=svc_grid.get_params(deep=True)['estimator__kernel'])

In [250]:
svc.fit(X_train, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [251]:
# Logging training parameters to AzureML and MLFlow experiments
run.log("C", svc_grid.get_params(deep=True)['estimator__C'])
run.log("Kernel", svc_grid.get_params(deep=True)['estimator__kernel'])

In [252]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [253]:
predicted_svc = svc.predict(X_test)

In [254]:
acc = accuracy_score(y_test, predicted_svc)

In [255]:
fscore = f1_score(y_test, predicted_svc, average="macro")
precision = precision_score(y_test, predicted_svc, average="macro")
recall = recall_score(y_test, predicted_svc, average="macro")

In [256]:
import git
repo = git.Repo(search_parent_directories=True)
sha = repo.head.object.hexsha

In [257]:
# Log to AzureML and MLflow
run.log("Test_accuracy", acc)
run.log("Precision", precision)
run.log("Recall", recall)
run.log("F-Score", fscore)
run.log("Git-sha", sha)

In [258]:
run.complete()
print ("run id:", run.id)

run id: 7b07ec3e-5750-4555-8f72-8dc332665152


In [259]:
mlflow.end_run()

🏃 View run musing_coconut_3nn6sbym at: https://eastus2.api.azureml.ms/mlflow/v2.0/subscriptions/f2f70602-65d9-479b-8014-a1c92a06ef0e/resourceGroups/Learn_MLOps/providers/Microsoft.MachineLearningServices/workspaces/MLOps/#/experiments/5e64a8df-a05c-4986-a21a-0d5f24b998ca/runs/c447bc73-d74a-494e-93b2-925978098359
🧪 View experiment at: https://eastus2.api.azureml.ms/mlflow/v2.0/subscriptions/f2f70602-65d9-479b-8014-a1c92a06ef0e/resourceGroups/Learn_MLOps/providers/Microsoft.MachineLearningServices/workspaces/MLOps/#/experiments/5e64a8df-a05c-4986-a21a-0d5f24b998ca


In [260]:
run.get_metrics()

{'C': 1.0,
 'Kernel': 'rbf',
 'Test_accuracy': 0.9519180922757906,
 'Precision': 0.8869828453699851,
 'Recall': 0.8859050416892464,
 'F-Score': 0.8864428755463127,
 'Git-sha': 'bc6cf950fb8a240586807928b03765fa1d3ac180'}

In [261]:
workspace.get_details()

{'id': '/subscriptions/f2f70602-65d9-479b-8014-a1c92a06ef0e/resourceGroups/Learn_MLOps/providers/Microsoft.MachineLearningServices/workspaces/MLOps',
 'name': 'MLOps',
 'identity': {'principal_id': '8d3fc917-4f4c-4fec-ade7-7a5ebf0f3610',
  'tenant_id': '5d23396c-0311-4c97-b944-3e4b1b6b92a7',
  'type': 'SystemAssigned'},
 'location': 'eastus2',
 'type': 'Microsoft.MachineLearningServices/workspaces',
 'tags': {},
 'sku': 'Basic',
 'workspaceid': 'd1a750b8-2231-411a-a393-1a93f2d63c15',
 'sdkTelemetryAppInsightsKey': 'e1f7b545-6243-4abf-ba76-c5691d2edb62',
 'description': '',
 'friendlyName': 'MLOps',
 'creationTime': '2026-05-05T16:03:09.1100285Z',
 'keyVault': '/subscriptions/f2f70602-65d9-479b-8014-a1c92a06ef0e/resourceGroups/Learn_MLOps/providers/Microsoft.Keyvault/vaults/mlops7586138321',
 'applicationInsights': '/subscriptions/f2f70602-65d9-479b-8014-a1c92a06ef0e/resourceGroups/Learn_MLOps/providers/Microsoft.insights/components/mlops6051692955',
 'storageAccount': '/subscriptions/f

In [262]:
import mlflow.sklearn
mlflow.sklearn.log_model(svc, 'outputs')

2026/05/05 18:07:17 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Random Forest classifier 

In [263]:
myexperiment = Experiment(workspace, "random-forest-classifier")
mlflow.set_experiment("mlflow-random-forest-classifier")

2026/05/05 18:07:41 INFO mlflow.tracking.fluent: Experiment with name 'mlflow-random-forest-classifier' does not exist. Creating a new experiment.


<Experiment: artifact_location='', creation_time=1778029661096, experiment_id='e59df1f6-1270-4ce5-b2fa-aa1c364a9c15', last_update_time=None, lifecycle_stage='active', name='mlflow-random-forest-classifier', tags={}>

In [264]:
rf = RandomForestClassifier(max_depth=10, random_state=0, n_estimators=100)

In [265]:
# initialize runs in Azureml and mlflow
run = myexperiment.start_logging()
mlflow.start_run()


# Log dataset used 
run.log("dataset name", dataset.name)
run.log("dataset Version", dataset.version)

Exception: Run with UUID 0c3cb7f6-b7ca-4538-9398-b769804c9e32 is already active. To start a new run, first end the current run with mlflow.end_run(). To start a nested run, call start_run with nested=True

In [266]:
%%time
rf.fit(X_train, y_train)

CPU times: user 7.21 s, sys: 4.06 ms, total: 7.21 s
Wall time: 7.21 s


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [267]:
# Logging training parameters to AzureML and MLFlow experiments
run.log("max_depth", 10)
run.log("random_state", 0)
run.log("n_estimators", 100)

In [268]:
predicted_rf = rf.predict(X_test)

In [269]:
acc = accuracy_score(y_test, predicted_rf)
fscore = f1_score(y_test, predicted_rf, average="macro")
precision = precision_score(y_test, predicted_rf, average="macro")
recall = recall_score(y_test, predicted_rf, average="macro")

In [270]:
run.log("Test_accuracy", acc)
run.log("Precision", precision)
run.log("Recall", recall)
run.log("F-Score", fscore)
run.log("Git-sha", sha)

In [271]:
run.complete()
print ("run id:", run.id)

run id: 2cadad7c-60c8-437f-9fec-65de1612db44


In [272]:
mlflow.end_run()

🏃 View run olden_glove_b2jk65r5 at: https://eastus2.api.azureml.ms/mlflow/v2.0/subscriptions/f2f70602-65d9-479b-8014-a1c92a06ef0e/resourceGroups/Learn_MLOps/providers/Microsoft.MachineLearningServices/workspaces/MLOps/#/experiments/5e64a8df-a05c-4986-a21a-0d5f24b998ca/runs/0c3cb7f6-b7ca-4538-9398-b769804c9e32
🧪 View experiment at: https://eastus2.api.azureml.ms/mlflow/v2.0/subscriptions/f2f70602-65d9-479b-8014-a1c92a06ef0e/resourceGroups/Learn_MLOps/providers/Microsoft.MachineLearningServices/workspaces/MLOps/#/experiments/5e64a8df-a05c-4986-a21a-0d5f24b998ca


In [273]:
run.get_metrics()

{'max_depth': 10,
 'random_state': 0,
 'n_estimators': 100,
 'Precision': 0.9021402611023651,
 'Recall': 0.8827541047507677,
 'Test_accuracy': 0.9553525142560912,
 'F-Score': 0.892111728299988,
 'Git-sha': 'bc6cf950fb8a240586807928b03765fa1d3ac180'}

# Model Packaging Step

pickle file or onnx

In [275]:
# Convert into SVC model into ONNX format file
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
initial_type = [('float_input', FloatTensorType([None, 6]))]
onx = convert_sklearn(svc, initial_types=initial_type)
with open("outputs/svc.onnx", "wb") as f:
    f.write(onx.SerializeToString())

In [276]:
# Convert into RF model into ONNX format file
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
initial_type = [('float_input', FloatTensorType([None, 6]))]
onx = convert_sklearn(rf, initial_types=initial_type)
with open("outputs/rf.onnx", "wb") as f:
    f.write(onx.SerializeToString())

# Model Registering Step

In [277]:
# Register Model on AzureML WS
model = Model.register(model_path = './outputs/svc.onnx', # this points to a local file 
                       model_name = "support-vector-classifier", # this is the name the model is registered as
                       tags = {'dataset': dataset.name, 'version': dataset.version, 'hyparameter-C': '1', 'testdata-accuracy': '0.9519'}, 
                       model_framework='pandas==0.23.4',
                       description = "Support vector classifier to predict weather at port of Turku",
                       workspace = workspace)

print('Name:', model.name)
print('Version:', model.version)

Registering model support-vector-classifier
Name: support-vector-classifier
Version: 1


In [278]:
# Register Model on AzureML WS
model = Model.register(model_path = './outputs/rf.onnx', # this points to a local file 
                       model_name = "random-forest-classifier", # this is the name the model is registered as
                       tags = {'dataset': dataset.name, 'version': dataset.version, 'hyparameter-C': '1', 'testdata-accuracy': '0.9548'}, 
                       model_framework='pandas==0.23.4',
                       description = "Random forest classifier to predict weather at port of Turku",
                       workspace = workspace)

print('Name:', model.name)
print('Version:', model.version)

Registering model random-forest-classifier
Name: random-forest-classifier
Version: 1


In [279]:
import mlflow.sklearn

In [280]:
# Save the model to the outputs directory for capture
mlflow.sklearn.log_model(svc, 'outputs/svc.onnx')

2026/05/05 18:12:59 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [281]:
# Save the model to the outputs directory for capture
mlflow.sklearn.log_model(rf, 'outputs/rf.onnx')

2026/05/05 18:13:15 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


# Save model artefacts

In [282]:
import pickle

with open('./outputs/scaler.pkl', 'wb') as scaler_pkl:
    pickle.dump(sc, scaler_pkl)

In [283]:
# Register Model on AzureML WS
scaler = Model.register(model_path = './outputs/scaler.pkl', # this points to a local file 
                       model_name = "scaler", # this is the name the model is registered as
                       tags = {'dataset': dataset.name, 'version': dataset.version}, 
                       model_framework='pandas==0.23.4',
                       description = "Scaler used for scaling incoming inference data",
                       workspace = workspace)

print('Name:', scaler.name)
print('Version:', scaler.version)

Registering model scaler
Name: scaler
Version: 1
